# Traffic Demand — Exploratory Data Analysis

This notebook explores the Flipkart Grid traffic demand dataset: distributions, missing values, temporal patterns, and geospatial structure.

In [ ]:
import sys
from pathlib import Path

import geohash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_train, load_test, split_train_holdout

sns.set_theme(style="whitegrid")
train = load_train()
test = load_test()
print("Train:", train.shape, "| Test:", test.shape)
train.head()

## Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train["demand"].hist(bins=60, ax=axes[0])
axes[0].set_title("Demand histogram")
axes[0].set_xlabel("demand")
np.log1p(train["demand"]).hist(bins=60, ax=axes[1])
axes[1].set_title("log1p(demand)")
plt.tight_layout()
plt.show()
print(train["demand"].describe())
print("Skew:", train["demand"].skew())

## Missing values & categoricals

In [ ]:
miss = train.isna().mean().sort_values(ascending=False)
print("Missing rate (train):\n", miss[miss > 0])

for col in ["RoadType", "Weather", "LargeVehicles", "Landmarks", "NumberofLanes"]:
    print(f"\n{col}:", train[col].value_counts(dropna=False).head())
print("\nUnique geohashes:", train["geohash"].nunique(), "| days:", sorted(train["day"].unique()))

## Temporal patterns

In [ ]:
hourly = train.groupby(["day", "hour"])["demand"].mean().reset_index()
plt.figure(figsize=(10, 4))
for day in sorted(train["day"].unique()):
    sub = hourly[hourly["day"] == day]
    plt.plot(sub["hour"], sub["demand"], marker="o", label=f"day {day}")
plt.xlabel("Hour")
plt.ylabel("Mean demand")
plt.title("Average demand by hour")
plt.legend()
plt.show()

print("Day 49 train timestamps:", sorted(train.loc[train["day"]==49, "timestamp"].unique()))
print("Test timestamps:", sorted(test["timestamp"].unique())[:5], "...", sorted(test["timestamp"].unique())[-3:])

## Geospatial layout (decoded geohash)

In [ ]:
coords = train["geohash"].map(lambda g: geohash.decode(g))
train_plot = train.copy()
train_plot["lat"] = coords.map(lambda x: x[0])
train_plot["lon"] = coords.map(lambda x: x[1])

geo_mean = train_plot.groupby("geohash").agg(lat=("lat", "first"), lon=("lon", "first"), demand=("demand", "mean")).reset_index()
plt.figure(figsize=(8, 6))
sc = plt.scatter(geo_mean["lon"], geo_mean["lat"], c=geo_mean["demand"], cmap="viridis", s=12, alpha=0.8)
plt.colorbar(sc, label="mean demand")
plt.title("Geohash locations colored by mean demand")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()

## Validation split note

Train day 49 only covers early morning (0:00–2:00). Test day 49 covers 2:15–13:45. We validate by training on **day 48** and predicting **day 49** rows in train.

In [ ]:
tr, va = split_train_holdout(train)
print(f"Train split: {len(tr)} rows (day {tr['day'].unique()})")
print(f"Val split: {len(va)} rows (day {va['day'].unique()})")
print(f"Test rows: {len(test)}")